In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps
import datetime
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

/var/folders/fb/2nwyf_mx609gfjxv_jjnb76h0000gn/T/ipykernel_13257/1857937564.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [410]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
         host="processing-2.processing.data.production.internal",
         
#         host="presto.processing.yoda.run",
        # host="processing.processing.data.production.internal",


        port=80,
        protocol='http',
        catalog ='hive',
        username='sameer.kumar@rapido.bike')
    

In [ ]:
strata_groups = []
count_group = "redeemId"
split_ratio = 0.5

# age = np.random.randint(low=18, high=60, size=1000)
# income = np.random.randint(low=18000, high=60000, size=1000)

# age_bucket = pd.qcut(x=age, q=[0,0.25,0.50,0.75,1],labels=["P<0.25","P0.25_0.5","P0.5_0.75","P_0.75+"])
# income_bucket = pd.qcut(x=income, q=[0,0.25,0.50,0.75,1],labels=["P<0.25","P0.25_0.5","P0.5_0.75","P_0.75+"])

# df = pd.DataFrame({'age':age,"income":income, "age_group": age_bucket, 'income_group': income_bucket})
# df

whole_set = df.index.values

np.random.seed(100)
stratified_sample = df.groupby(strata_groups).apply(lambda x: x.sample(frac=split_ratio,random_state=100))
strata= stratified_sample.droplevel(list(range(len(strata_groups))))

test_group = np.sort(strata.index.values)
control_group = np.sort(np.delete(arr=whole_set, obj=test_group,axis=None))

test_df = df.iloc[test_group]
control_df = df.iloc[control_group]
test_df.shape, control_df.shape

def split_check_func(split_input_df, data_type):
    split_check = split_input_df.groupby(strata_groups).agg({count_group: ['count']})
    for i in range(len(strata_groups)):
        split_check.insert(loc=i,column= split_check.index.names[i], value=split_check.index.get_level_values(i))
    split_check = split_check.reset_index(drop=True)
    split_check.columns= ["_".join(split_check.columns[i]).rstrip("_") for i in range(len(split_check.columns))]
    column_name = [i for i in split_check.columns if "_count" in i]
    new_column = data_type+"_"+column_name[0]
    print(column_name, new_column)
    split_check = split_check.rename(columns={column_name[0]: new_column})
    print(data_type,split_check[new_column].sum())
    split_check[data_type+'_perc'] = split_check[new_column]/ np.sum(split_check[new_column])
    return(split_check)

test_split = split_check_func(test_df, "test")
control_split = split_check_func(control_df,"control")

test_control_df = pd.merge(left=test_split, right=control_split, on= strata_groups, how='inner')
# test_control_df.to_clipboard()

test_col = 'test_{}_count'.format(count_group)
control_col = 'control_{}_count'.format(count_group)

test_control_df["split_ratio_check"] = np.round(test_control_df[test_col]/(test_control_df[test_col]+ test_control_df[control_col]),2)
print(np.quantile(test_control_df['split_ratio_check'], q=[0.05,0.25,0.5,0.75,0.95]))
test_control_df


In [7]:
df=pd.read_sql('select * from captain.captain_transactions_snapshot limit 10',metabase_connection)

In [132]:
df=pd.read_clipboard()

In [ ]:
q="""
   with bot_initiated as(
  select event_props_rider_id as rider_id,event_props_context as bot_context,event_props_service as service,event_props_city as city,count(event_props_rider_id) as totalClicks
  from clevertap.captain_bot_initiated_immutable
  where yyyymmdd between '20250206' and '20250223'
  group by 1,2,3,4
),
bot_csat as(
   select event_props_rider_id as rider_id,event_props_context as bot_context,json_extract_scalar(event_props,'$.city') as city,
  json_extract_scalar(event_props,'$.service') as service,
  count(event_props_rider_id) as totalRatings,
  count(case when event_props_csat='Satisfied' then event_props_rider_id end) as SatisfiedCount,
  count(case when event_props_csat='Not Satisfied' then event_props_rider_id end) as UnSatisfiedCount,
  count(case when event_props_csat='' then event_props_rider_id end) as Not_rated_count
  from  clevertap.captain_bot_csat_immutable
 where yyyymmdd between '20250206' and '20250223'
  group by 1,2,3,4
),
finalTbl as (select a.*,b.totalRatings,b.SatisfiedCount,b.UnSatisfiedCount,b.Not_rated_count
from bot_initiated a
left join bot_csat b
on a.rider_id=b.rider_id
and a.bot_context=b.bot_context
and a.service=b.service
and a.city=b.city)
select city,service,bot_context,count(distinct rider_id) as distinct_captains,sum(totalClicks) as totalClicks,sum(totalRatings) as totalRatingsProvided,sum(SatisfiedCount) as totalSatisfied,
sum(UnSatisfiedCount) as totalUnsatisfied,sum(Not_rated_count) as totalNotRated
from finalTbl
group by 1,2,3
"""
df=pd.read_sql(q,metabase_connection)

DatabaseError: Execution failed on sql: 
   with bot_initiated as(
  select yyyymmdd,count(event_props_rider_id) as totalClicks
  from clevertap.captain_bot_initiated_immutable
  where yyyymmdd between '20250206' and '20250223'
  group by 1
),
bot_csat as(
   select yyyymmdd,
  count(event_props_rider_id) as totalRatings,
  count(case when event_props_csat='Satisfied' then event_props_rider_id end) as SatisfiedCount,
  count(case when event_props_csat='Not Satisfied' then event_props_rider_id end) as UnSatisfiedCount,
  count(case when event_props_csat='' then event_props_rider_id end) as Not_rated_count
  from  clevertap.captain_bot_csat_immutable
 where yyyymmdd between '20250206' and '20250223'
  group by 1
),
finalTbl as (select a.*,b.totalRatings,b.SatisfiedCount,b.UnSatisfiedCount,b.Not_rated_count
from bot_initiated a
left join bot_csat b
on a.yyyymmdd=b.yyyymmdd)
select yyyymmdd,count(distinct rider_id) as distinct_captains,sum(totalClicks) as totalClicks,sum(totalRatings) as t

In [406]:
df=df[df.bot_context!='']

In [407]:
df.to_clipboard()

In [388]:
q=""" select event_props_rider_id as rider_id,event_props_context as bot_context,event_props_service as service,event_props_city as city,count(event_props_rider_id) as totalClicks
  from clevertap.captain_bot_initiated_immutable
  where yyyymmdd between '20250206' and '20250223'
  group by 1,2,3,4
  """
df2=pd.read_sql(q,metabase_connection)

In [390]:
df2[df2.rider_id=='5d61590cd0286d106d61f1e3']

,rider_id,bot_context,service,city,totalClicks
146348,5d61590cd0286d106d61f1e3,bot_context__bob,Zepto-Darkstore,Bangalore,1
189769,5d61590cd0286d106d61f1e3,bot_context__bob,Link,Bangalore,133
248997,5d61590cd0286d106d61f1e3,bot_context__bob,C2C,Bangalore,1
296833,5d61590cd0286d106d61f1e3,bot_context__bob,Bike Lite,Bangalore,8
340555,5d61590cd0286d106d61f1e3,bot_context__kmadjustment,Bike Taxi,Bangalore,3


In [ ]:
dftemp[dftemp.rider_id=='5d61590cd0286d106d61f1e3']

,rider_id,bot_context,city,service,totalRatings,SatisfiedCount,UnSatisfiedCount,Not_rated_count
28513,5d61590cd0286d106d61f1e3,bot_context__bob,Bangalore,Link,28,24,4,0
72177,5d61590cd0286d106d61f1e3,bot_context__bob,Bangalore,Bike Lite,1,0,1,0


In [65]:
df2.columns=['mobilenumber']

In [393]:
botJoins=pd.merge(df2,dftemp,how='left',on=['rider_id','service','city','bot_context'])

In [137]:
finalDf=pd.concat([finalDf,df],axis=0).reset_index(drop='index')

In [399]:
df[df.rider_id=='5d61590cd0286d106d61f1e3']

,rider_id,bot_context,service,city,totalClicks,totalRatings,SatisfiedCount,UnSatisfiedCount,Not_rated_count
96820,5d61590cd0286d106d61f1e3,bot_context__bob,Bike Lite,Bangalore,8,1.0,0.0,1.0,0.0
115131,5d61590cd0286d106d61f1e3,bot_context__kmadjustment,Bike Taxi,Bangalore,3,NaN,NaN,NaN,NaN
229824,5d61590cd0286d106d61f1e3,bot_context__bob,Link,Bangalore,133,28.0,24.0,4.0,0.0
318598,5d61590cd0286d106d61f1e3,bot_context__bob,C2C,Bangalore,1,NaN,NaN,NaN,NaN
423413,5d61590cd0286d106d61f1e3,bot_context__bob,Zepto-Darkstore,Bangalore,1,NaN,NaN,NaN,NaN


In [ ]:
df

,rider_id,bot_context,service,city,totalClicks
0,62249e9d2bcb6b4d989ab496,bot_context__bob,Adloggs-ONDC,Pune,3
1,60e3c7d409aa99608940ac80,bot_context__bob,Link,Indore,7
2,65e49195d4428d2db22a7d6b,bot_context__bob,Link,Chennai,2
3,63070582e3fa3c49fd52194c,bot_context__bob,Link,Bangalore,101
4,660f9c0fd3bd693167644eb3,bot_context__bob,Link,Bangalore,102
...,...,...,...,...,...
444475,5c8a21f28c352421eae9873a,bot_context__suspension,,Bangalore,1
444476,5da1b09da1f0a71c606586a5,,Link,Kadapa,2
444477,5c0261540efaa90c32d74798,bot_context__bob,Link,Ahmedabad,2
444478,677d3cfa1c7f1786de3408e8,bot_context__bob,Link,Jaipur,100


In [94]:
finalDf=finalDf[['mobilenumber']]

In [96]:
finalDf.dropna(inplace=True)

In [144]:
finalDf=finalDf.astype({'mobilenumber':'str'})

In [146]:
finalDf.to_pickle('experiment3base.pkl')

In [ ]:
finalDf.drop_duplicates(inplace=True)

,mobilenumber
0,9710722789
1,9952004603
2,7200959431
3,9629477782
4,7339586687
...,...
531926,8939734144
531927,9597554703
531928,6379952114
531929,9597915587


In [305]:
q="""
    select captainid captain_id, mobilenumber
    from datasets.captain_single_view
    where lower(lastridecity)='chennai'
"""
captainphone=pd.read_sql(q,metabase_connection)

In [306]:
q="""
  select 
            distinct captain_id, performance_segment, consistency_segment
        from datasets_internal.captain_subservice_segments_v1
        where lower(service_mode)= 'all'
        and lower(service) = 'link'
        and lower(city) in ('chennai')
        and run_date = '20250223'
"""
segment=pd.read_sql(q,metabase_connection)

In [307]:
q="""
 select distinct captain_id,gig_segment
      from ( 
            select *
            from reports_internal.captain_gig_appography_v4
            where run_date = '20250107'
            and lower(city) in ('chennai')
          )
"""
gigStatus=pd.read_sql(q,metabase_connection)

In [250]:
gigStatus

,captain_id,gig_segment
0,5d8b1122f354c15d38cbd392,10_RHA_gig
1,652f8070d931a460b66fb728,01_nonRHA_gig
2,617a8863bc0934cb9368b778,10_RHA_gig
3,6204c70793d3530a87ad5928,10_RHA_gig
4,60430990103d600be5172db2,00_non_Gig
...,...,...
606919,5d138f79b1bdeb340ea27d19,01_nonRHA_gig
606920,5c9531509d2fd62fdb80ca92,00_non_Gig
606921,62ab2b5bf663e1b671b9ed52,10_RHA_gig
606922,62f87ab93829245ac7e240be,10_RHA_gig


In [308]:
q="""
    select captain_id,sum(net_rides_taxi) as net_rides, sum(total_lh) as total_lh, sum(final_captain_earnings) as total_earnings, sum(total_incentives) as total_incentives,count(case when net_rides_taxi>0 then yyyymmdd end) as net_days
   from reports_internal.consideration_temp_captain_logs_base
   where yyyymmdd between '20250208' and '20250222'
   and lower(city) in ('chennai')
   group by 1
"""
chennaiData=pd.read_sql(q,metabase_connection)


In [309]:
chennaiData=chennaiData[chennaiData.net_rides>0]

In [310]:
chennaiData

,captain_id,net_rides,total_lh,total_earnings,total_incentives,net_days
0,5f6c97c57d99a7316955b47e,20.0,21.9057,2049.250809,0.0,7
1,63776782f5ad695bc90368aa,14.0,6.2067,643.916590,45.0,3
3,64f6c8d7e2f8b54035489f85,60.0,22.7760,2908.296054,82.0,5
4,60f66044a0d79f1756bae4d6,9.0,5.2295,256.797901,0.0,3
6,61fa475f5d5c5bf85014af86,24.0,6.8455,893.952331,258.0,3
...,...,...,...,...,...,...
118517,66c8ae7d9d865264e576b0f8,4.0,1.2005,175.777771,0.0,3
118518,5cfdb6a5ca6e292111784c67,5.0,1.9865,214.482577,0.0,1
118542,62eca2e08fa0216a9c9d53e6,1.0,1.3761,49.770070,0.0,1
118556,60a9232cccc177f7b5a6cbaa,3.0,1.6288,138.727225,40.0,1


In [300]:
chennaiData=pd.merge(chennaiData,gigStatus,how='left',on='captain_id')

In [324]:
chennaiData

,captain_id,net_rides,total_lh,total_earnings,total_incentives,net_days,mobilenumber,performance_segment,consistency_segment
0,5f6c97c57d99a7316955b47e,20.0,21.9057,2049.250809,0.0,7,8778945106,LP,Inter
1,63776782f5ad695bc90368aa,14.0,6.2067,643.916590,45.0,3,6382242814,LP,Sporadic
2,64f6c8d7e2f8b54035489f85,60.0,22.7760,2908.296054,82.0,5,9962709055,HP,Intra
3,60f66044a0d79f1756bae4d6,9.0,5.2295,256.797901,0.0,3,9677202717,LP,Sporadic
4,61fa475f5d5c5bf85014af86,24.0,6.8455,893.952331,258.0,3,9080582128,HP,Intra
...,...,...,...,...,...,...,...,...,...
54701,66c8ae7d9d865264e576b0f8,4.0,1.2005,175.777771,0.0,3,9360489100,LP,Sporadic
54702,5cfdb6a5ca6e292111784c67,5.0,1.9865,214.482577,0.0,1,6388576804,LP,Sporadic
54703,62eca2e08fa0216a9c9d53e6,1.0,1.3761,49.770070,0.0,1,7397606687,LP,Intra
54704,60a9232cccc177f7b5a6cbaa,3.0,1.6288,138.727225,40.0,1,9600574538,LP,Sporadic


In [325]:
chennaiData=chennaiData[chennaiData.consistency_segment=='Sporadic']

In [327]:
chennaiData.reset_index(drop='index',inplace=True)

In [312]:
chennaiData=pd.merge(chennaiData,segment,how='left',on='captain_id')

In [258]:
chennaiData=pd.merge(finalDf,chennaiData,how='inner',on='mobilenumber')

In [311]:
chennaiData=pd.merge(chennaiData,captainphone,how='left',on='captain_id')

In [331]:
df=pd.merge(chennaiData,finalDf,how='inner',on='mobilenumber')

In [272]:
chennaiData[['mobilenumber','captain_id']].nunique()
chennaiData.drop_duplicates(subset=['captain_id','mobilenumber'],inplace=True)

In [336]:
df=df.drop_duplicates(subset=['captain_id','mobilenumber']).reset_index(drop='index')

In [337]:
df

,captain_id,net_rides,total_lh,total_earnings,total_incentives,net_days,mobilenumber,performance_segment,consistency_segment
0,63776782f5ad695bc90368aa,14.0,6.2067,643.916590,45.0,3,6382242814,LP,Sporadic
1,60f66044a0d79f1756bae4d6,9.0,5.2295,256.797901,0.0,3,9677202717,LP,Sporadic
2,61fcdff8540ed824b8433411,1.0,1.5192,90.029377,0.0,1,8939589455,LP,Sporadic
3,5f27f70f4b3a15c2bcc52c6c,4.0,5.7921,554.797869,31.0,2,8825507630,LP,Sporadic
4,5f90017b685417b57374bc96,2.0,8.7386,553.375669,0.0,2,9042302152,LP,Sporadic
...,...,...,...,...,...,...,...,...,...
19145,66509ab13b221e7c81ed9337,11.0,8.5254,687.530714,0.0,2,8939636261,MP,Sporadic
19146,6430fad32548e39ae4e535d7,13.0,1.9604,386.069048,0.0,2,8056157395,MP,Sporadic
19147,6712dd0ee8a57d21adde6585,17.0,7.5448,990.772196,0.0,5,8428681772,LP,Sporadic
19148,66c8ae7d9d865264e576b0f8,4.0,1.2005,175.777771,0.0,3,9360489100,LP,Sporadic


In [274]:
chennaiData.reset_index(drop='index',inplace=True)

In [346]:
df['net_rides_cat']=pd.qcut(df['net_rides'],3,['A','B','C'])

In [347]:
df['total_lh_cat']=pd.qcut(df['total_lh'],3,['D','E','F'])
df['total_earnings_cat']=pd.qcut(df['total_earnings'],3,['H','I','J'])
df['net_days_cat']=pd.qcut(df['net_days'],3,['K','L'],duplicates='drop')
# chennaiData['total_incentives_cat']=pd.qcut(chennaiData['total_incentives'],3,['N','O','P'])

In [348]:
df['total_incentives_cat']=pd.qcut(df['total_incentives'],3,['N','M'],duplicates='drop')

In [351]:
df=pd.merge(df,gigStatus,how='left',on='captain_id')

In [356]:
df.gig_segment.fillna('NA',inplace=True)

In [357]:
df

,captain_id,net_rides,total_lh,total_earnings,total_incentives,net_days,mobilenumber,performance_segment,consistency_segment,net_rides_cat,total_lh_cat,total_earnings_cat,net_days_cat,total_incentives_cat,gig_segment
0,63776782f5ad695bc90368aa,14.0,6.2067,643.916590,45.0,3,6382242814,LP,Sporadic,C,E,I,K,M,01_nonRHA_gig
1,60f66044a0d79f1756bae4d6,9.0,5.2295,256.797901,0.0,3,9677202717,LP,Sporadic,B,E,I,K,N,00_non_Gig
2,61fcdff8540ed824b8433411,1.0,1.5192,90.029377,0.0,1,8939589455,LP,Sporadic,A,D,H,K,N,11_Full_Gig
3,5f27f70f4b3a15c2bcc52c6c,4.0,5.7921,554.797869,31.0,2,8825507630,LP,Sporadic,B,E,I,K,M,11_Full_Gig
4,5f90017b685417b57374bc96,2.0,8.7386,553.375669,0.0,2,9042302152,LP,Sporadic,A,E,I,K,N,11_Full_Gig
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19145,66509ab13b221e7c81ed9337,11.0,8.5254,687.530714,0.0,2,8939636261,MP,Sporadic,C,E,J,K,N,10_RHA_gig
19146,6430fad32548e39ae4e535d7,13.0,1.9604,386.069048,0.0,2,8056157395,MP,Sporadic,C,D,I,K,N,11_Full_Gig
19147,6712dd0ee8a57d21adde6585,17.0,7.5448,990.772196,0.0,5,8428681772,LP,Sporadic,C,E,J,L,N,00_non_Gig
19148,66c8ae7d9d865264e576b0f8,4.0,1.2005,175.777771,0.0,3,9360489100,LP,Sporadic,B,D,H,K,N,00_non_Gig


In [358]:
strata_groups = ['gig_segment', 'performance_segment', 'net_rides_cat', 'total_lh_cat','total_earnings_cat', 'net_days_cat']
count_group = "captain_id"
split_ratio = 0.5

# age = np.random.randint(low=18, high=60, size=1000)
# income = np.random.randint(low=18000, high=60000, size=1000)

# age_bucket = pd.qcut(x=age, q=[0,0.25,0.50,0.75,1],labels=["P<0.25","P0.25_0.5","P0.5_0.75","P_0.75+"])
# income_bucket = pd.qcut(x=income, q=[0,0.25,0.50,0.75,1],labels=["P<0.25","P0.25_0.5","P0.5_0.75","P_0.75+"])

# df = pd.DataFrame({'age':age,"income":income, "age_group": age_bucket, 'income_group': income_bucket})
# df

whole_set = df.index.values

np.random.seed(100)
stratified_sample = df.groupby(strata_groups).apply(lambda x: x.sample(frac=split_ratio,random_state=100))
strata= stratified_sample.droplevel(list(range(len(strata_groups))))

test_group = np.sort(strata.index.values)
control_group = np.sort(np.delete(arr=whole_set, obj=test_group,axis=None))

test_df = df.iloc[test_group]
control_df = df.iloc[control_group]
test_df.shape, control_df.shape

def split_check_func(split_input_df, data_type):
    split_check = split_input_df.groupby(strata_groups).agg({count_group: ['count']})
    for i in range(len(strata_groups)):
        split_check.insert(loc=i,column= split_check.index.names[i], value=split_check.index.get_level_values(i))
    split_check = split_check.reset_index(drop=True)
    split_check.columns= ["_".join(split_check.columns[i]).rstrip("_") for i in range(len(split_check.columns))]
    column_name = [i for i in split_check.columns if "_count" in i]
    new_column = data_type+"_"+column_name[0]
    print(column_name, new_column)
    split_check = split_check.rename(columns={column_name[0]: new_column})
    print(data_type,split_check[new_column].sum())
    split_check[data_type+'_perc'] = split_check[new_column]/ np.sum(split_check[new_column])
    return(split_check)

test_split = split_check_func(test_df, "test")
control_split = split_check_func(control_df,"control")

test_control_df = pd.merge(left=test_split, right=control_split, on= strata_groups, how='inner')
# test_control_df.to_clipboard()

test_col = 'test_{}_count'.format(count_group)
control_col = 'control_{}_count'.format(count_group)

test_control_df["split_ratio_check"] = np.round(test_control_df[test_col]/(test_control_df[test_col]+ test_control_df[control_col]),2)
print(np.quantile(test_control_df['split_ratio_check'], q=[0.05,0.25,0.5,0.75,0.95]))
test_control_df


['captain_id_count'] test_captain_id_count
test 9561
['captain_id_count'] control_captain_id_count
control 9589
[nan nan nan nan nan]


,gig_segment,performance_segment,net_rides_cat,total_lh_cat,total_earnings_cat,net_days_cat,test_captain_id_count,test_perc,control_captain_id_count,control_perc,split_ratio_check
0,00_non_Gig,HP,A,D,H,K,1,0.000105,1,0.000104,0.5
1,00_non_Gig,HP,A,D,H,L,0,0.000000,0,0.000000,NaN
2,00_non_Gig,HP,A,D,I,K,0,0.000000,0,0.000000,NaN
3,00_non_Gig,HP,A,D,I,L,0,0.000000,0,0.000000,NaN
4,00_non_Gig,HP,A,D,J,K,0,0.000000,0,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1075,NA,UHP,C,F,H,L,0,0.000000,0,0.000000,NaN
1076,NA,UHP,C,F,I,K,0,0.000000,0,0.000000,NaN
1077,NA,UHP,C,F,I,L,0,0.000000,0,0.000000,NaN
1078,NA,UHP,C,F,J,K,3,0.000314,3,0.000313,0.5


In [359]:
test_df=test_df[['mobilenumber','gig_segment','performance_segment','consistency_segment']]

In [360]:
control_df=control_df[['mobilenumber','gig_segment','performance_segment','consistency_segment']]

In [233]:
df=df.astype(str)

In [361]:
test_df.reset_index(drop='Index',inplace=True)
control_df.reset_index(drop='Index',inplace=True)

In [ ]:
df['concatenated'] = df.apply(lambda row: ''.join(row), axis=1)

In [362]:
test_df.to_csv('chennai_test_df.csv')
control_df.to_csv('chennai_control_df.csv')

In [363]:
test_df

,mobilenumber,gig_segment,performance_segment,consistency_segment
0,6382242814,01_nonRHA_gig,LP,Sporadic
1,8610509343,NA,MP,Sporadic
2,9150121140,00_non_Gig,LP,Sporadic
3,7402214035,11_Full_Gig,LP,Sporadic
4,9087027211,01_nonRHA_gig,MP,Sporadic
...,...,...,...,...
9556,9677240857,10_RHA_gig,LP,Sporadic
9557,9500516642,00_non_Gig,LP,Sporadic
9558,8939636261,10_RHA_gig,MP,Sporadic
9559,8428681772,00_non_Gig,LP,Sporadic


In [364]:
control_df

,mobilenumber,gig_segment,performance_segment,consistency_segment
0,9677202717,00_non_Gig,LP,Sporadic
1,8939589455,11_Full_Gig,LP,Sporadic
2,8825507630,11_Full_Gig,LP,Sporadic
3,9042302152,11_Full_Gig,LP,Sporadic
4,9003148698,11_Full_Gig,LP,Sporadic
...,...,...,...,...
9584,7358680346,01_nonRHA_gig,LP,Sporadic
9585,7200009673,00_non_Gig,MP,Sporadic
9586,9003181123,00_non_Gig,MP,Sporadic
9587,8056157395,11_Full_Gig,MP,Sporadic


In [419]:
q="""
   
with bot_initiated as(
  select event_props_city as city,count(event_props_rider_id) as totalClicks
  from clevertap.captain_bot_initiated_immutable
  where yyyymmdd between '20250206' and '20250223'
  group by 1
),
bot_csat as(
   select json_extract_scalar(event_props,'$.city') as city,
  count(event_props_rider_id) as bot_totalRatings,
  count(case when event_props_csat='Satisfied' then event_props_rider_id end) as bot_SatisfiedCount,
  count(case when event_props_csat='Not Satisfied' then event_props_rider_id end) as bot_UnSatisfiedCount,
  count(case when event_props_csat='' then event_props_rider_id end) as bot_Not_rated_count
  from  clevertap.captain_bot_csat_immutable
 where yyyymmdd between '20250206' and '20250223'
  group by 1
),calling_data as(
        select event_props_city as city,count(distinct profile_identity) totalDistinctCalledCaps,count(profile_identity) totalCalls
        from  clevertap.captain_call_support_immutable
        where yyyymmdd between '20250206' and '20250223'
        group by 1
 ),
    rating_data as(
    select json_extract_scalar(event_props,'$.city') as city,count(distinct profile_identity) as CallingtotalDistinctRatedCaps,count(profile_identity) as CallingtotalRatingGiven,
    count(case when event_props_csat='Satisfied' then profile_identity end) as CallingSatisfied,
    count(case when event_props_csat='Not Satisfied' then profile_identity end) as CallingnotSatisfied,
    count(case when event_props_csat='' then profile_identity end) as CallingnotRated
    from clevertap.captain_bot_call_support_immutable
    group by 1
)
select a.*,b.bot_totalRatings,b.bot_SatisfiedCount,b.bot_UnSatisfiedCount,b.bot_Not_rated_count,d.totalCalls,c.CallingtotalDistinctRatedCaps,
c.CallingtotalRatingGiven,c.CallingSatisfied,
c.CallingnotSatisfied,c.CallingnotRated
from bot_initiated a
left join calling_data d
on a.city=d.city
left join bot_csat b
on a.city=b.city
left join rating_data c
on a.city=c.city 
"""
df=pd.read_sql(q,metabase_connection)

In [421]:
df.to_clipboard()